# Waze: ¿le gana un modelo de árbol a la regresión?

**Curso 5 del certificado, proyecto de modelos basados en árboles.**

El Curso 4 ya construyó un modelo para esto y dejó sus números escritos. Así que este
proyecto puede hacer algo que ningún proyecto suelto puede: **comparar de verdad**.

La partición está montada para eso. El primer corte es el mismo 25 % con la misma semilla,
o sea que **los 3.575 usuarios de prueba son literalmente los mismos**, y los 10.724 que el
Curso 4 usó enteros para entrenar aquí se reparten entre entrenamiento y validación.

**Lo que espero, escrito antes de ajustar nada:** que los árboles no ganen. Una comparación
sin ajustar ya lo sugería. Este cuaderno los ajusta en serio para que la respuesta valga en
las dos direcciones.

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "projects" / "curso5").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "projects"))
sys.path.insert(0, str(ROOT / "projects" / "curso5" / "waze" / "02_scripts"))

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from waze_trees import FEATURES, SEED, SEGMENT_COLUMNS, load, split
from common import Results, dataset

pd.set_option("display.width", 120)
print("listo")

listo


## 1. Los datos, preparados exactamente igual que en el Curso 4

Esto no es pereza: cambiar una sola variable construida invalidaría la comparación, que es
lo único que este proyecto viene a hacer.

In [2]:
results = Results("waze", "Waze, cuaderno del Curso 5", "curso5")
df = load(results)
train, validation, test = split(df, results)


1. Los datos, preparados igual que en el Curso 4
  filas en el CSV: 14999
  usuarios sin etiqueta: 700
  usuarios con etiqueta: 14299
  tasa de abandono (%): 17.74

2. La partición en tres, con la prueba intacta hasta el final
    abandono en train (%): 17.74
    abandono en validation (%): 17.72
    abandono en test (%): 17.73
  los que el Curso 4 usó enteros para entrenar: 10724
  entrenamiento 8579   validación 2145   prueba 3575
  la prueba es la misma que la del Curso 4: 3575 usuarios


## 2. La partición en tres, y por qué no en dos

El conjunto de validación se gasta: cada vez que miras un resultado y cambias algo, le
filtras información. Después de comparar cuatro familias y probar 88 combinaciones de
perillas, ya no es una medida independiente.

La prueba no participa en ninguna decisión, y por eso es la única cifra que se publica.

In [3]:
for name, part in [("entrenamiento", train), ("validacion", validation), ("prueba", test)]:
    print(f"{name:15} {len(part):6} usuarios   abandono {100*part.churned.mean():.2f} %")
print()
print(f"la prueba del Curso 4 tenia {len(test)} usuarios, y son estos mismos")

entrenamiento     8579 usuarios   abandono 17.74 %
validacion        2145 usuarios   abandono 17.72 %
prueba            3575 usuarios   abandono 17.73 %

la prueba del Curso 4 tenia 3575 usuarios, y son estos mismos


## 3. Agrupar sin etiquetas, y el número de grupos elegido con la silueta

Módulo 3 aplicado a usuarios de verdad. La pregunta difícil no es cómo agrupar: es cuántos
grupos hay, y eso no lo calcula el algoritmo.

In [4]:
scaler = StandardScaler().fit(train[SEGMENT_COLUMNS])
scaled = scaler.transform(train[SEGMENT_COLUMNS])

for k in range(2, 8):
    model = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(scaled)
    print(f"k={k}   inercia {model.inertia_:9.0f}   "
          f"silueta {silhouette_score(scaled, model.labels_):.4f}")

k=2   inercia     33634   silueta 0.2588
k=3   inercia     28492   silueta 0.1937
k=4   inercia     24448   silueta 0.2061
k=5   inercia     21502   silueta 0.2021
k=6   inercia     19573   silueta 0.1957
k=7   inercia     18217   silueta 0.1919


**La silueta no llega a 0,26 en ningún caso.** Por debajo de 0,35 los grupos se tocan entre
sí, así que estos usuarios no forman familias separadas: son un continuo.

Se reporta así. K-medias siempre devuelve tantos grupos como le pidas, también sobre datos
sin ninguna estructura, y la silueta es lo que distingue un hallazgo de un reparto
arbitrario.

In [5]:
km = KMeans(n_clusters=2, n_init=20, random_state=SEED).fit(scaled)
for part in (train, validation, test):
    part["segment"] = km.predict(scaler.transform(part[SEGMENT_COLUMNS]))

print("abandono por segmento, en entrenamiento")
print((train.groupby("segment").churned.agg(["size", "mean"])
       .assign(mean=lambda d: (100 * d["mean"]).round(2))).to_string())

abandono por segmento, en entrenamiento
         size   mean
segment             
0        6395  17.29
1        2184  19.05


1,75 puntos de diferencia entre los dos grupos. **No es una segmentación accionable**, y
aun así la variable entra en el modelo, para que sea el modelo quien lo diga.

![La segmentación, que no encontró grupos](03_figures/01_segmentos.png)

## 4. Cuatro candidatos, medidos en validación

Sin tocar la prueba. Un árbol sin acotar se incluye a propósito, para verlo perder.

In [6]:
X, y = train[FEATURES], train.churned
Xv, yv = validation[FEATURES], validation.churned

from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

for name, model in {
    "logistica": LogisticRegression(max_iter=3000),
    "arbol":     DecisionTreeClassifier(random_state=SEED),
    "bosque":    RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    "xgboost":   XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                               eval_metric="logloss", random_state=SEED, n_jobs=1),
}.items():
    model.fit(X, y)
    print(f"{name:11} AUC {roc_auc_score(yv, model.predict_proba(Xv)[:, 1]):.4f}")

logistica   AUC 0.7424
arbol       AUC 0.5717
bosque      AUC 0.7129
xgboost     AUC 0.7273


El árbol suelto saca 0,5717, que está a un paso de tirar una moneda. Ese es el punto de
partida de las dos familias que vienen: **no se juzga un bosque por lo que hace un árbol.**

## 5. Ajustar las perillas, para que ninguna familia pierda por desatendida

Con validación cruzada de cuatro pliegues dentro del entrenamiento. Los resultados completos
están en el guion; aquí se muestra el bosque, que es el que más gana.

In [7]:
forest = RandomForestClassifier(n_estimators=300, max_depth=6, max_features=0.5,
                                min_samples_leaf=50, random_state=SEED, n_jobs=-1).fit(X, y)
logistic = LogisticRegression(max_iter=3000).fit(X, y)

print(f"bosque sin ajustar   0.7129")
print(f"bosque ajustado      {roc_auc_score(yv, forest.predict_proba(Xv)[:, 1]):.4f}")
print(f"logistica            {roc_auc_score(yv, logistic.predict_proba(Xv)[:, 1]):.4f}")

bosque sin ajustar   0.7129
bosque ajustado      0.7405
logistica            0.7424


**Ajustar el bosque le dio 0,0276 de AUC y se quedó a 0,0019 de la logística**, que no lleva
ninguna perilla que ajustar.

Ese es el resultado del proyecto dicho en una línea: afinar el modelo complicado casi cierra
la distancia, y no la cierra.

![Lo que compró el ajuste](03_figures/02_ajuste.png)

## 6. Qué usó el bosque, y qué no

La importancia de variables es lo que devuelve un bosque al terreno de lo explicable.

In [8]:
order = np.argsort(forest.feature_importances_)[::-1]
for position in order:
    print(f"{FEATURES[position]:26} {forest.feature_importances_[position]:.4f}")

activity_days              0.5163
n_days_after_onboarding    0.1687
km_per_driving_day         0.1594
drives                     0.0497
duration_minutes_drives    0.0445
total_sessions             0.0343
professional_driver        0.0216
iphone                     0.0030
segment                    0.0026


**Los días activo son más de la mitad del modelo.** Y el segmento del módulo 3 queda el
último de nueve, con 0,0026: la silueta ya avisaba de que no había grupos y el modelo lo
confirma.

![La importancia, con el segmento el último](03_figures/03_importancia.png)

## 7. La prueba, una sola vez

Después de haber elegido campeón en validación. Se miden los tres finalistas para poder
contarlo, pero la decisión ya estaba tomada.

In [9]:
Xt, yt = test[FEATURES], test.churned
from sklearn.metrics import accuracy_score, recall_score

for name, model in {"logistica": logistic, "bosque ajustado": forest}.items():
    p = model.predict_proba(Xt)[:, 1]
    d = (p >= 0.5).astype(int)
    print(f"{name:16} AUC {roc_auc_score(yt, p):.4f}   "
          f"exactitud {accuracy_score(yt, d):.4f}   "
          f"sensibilidad {recall_score(yt, d):.4f}")
print()
print("el Curso 4 publico:  AUC 0.7368   exactitud 0.8229   sensibilidad 0.0820")

logistica        AUC 0.7379   exactitud 0.8249   sensibilidad 0.0962
bosque ajustado  AUC 0.7318   exactitud 0.8243   sensibilidad 0.0710

el Curso 4 publico:  AUC 0.7368   exactitud 0.8229   sensibilidad 0.0820


![La comparación entre cursos](03_figures/04_comparacion.png)

## 8. La respuesta

**No, el modelo complicado no gana.** Sobre los mismos 3.575 usuarios, la logística saca
0,7379 y los dos árboles ajustados 0,7318 y 0,7348.

Y hay un segundo hallazgo que sobrevive a los cuatro modelos: **ninguno detecta ni a uno de
cada diez de los que se van**. El Curso 4 ya había diagnosticado que eso era el umbral y no
el modelo, y esta es la confirmación desde otro sitio: con tres modelos más, la sensibilidad
sigue clavada.

![La sensibilidad, que no se movió](03_figures/05_sensibilidad.png)

**Lo que se decide:** desplegar la logística, que gana y además se explica; bajar el umbral,
que es lo único que mueve la sensibilidad; y no usar la segmentación para dirigir campañas.

**Lo que no se puede afirmar:** que la diferencia de 0,0031 de AUC entre la logística y el
refuerzo sea real. Con 3.575 usuarios eso no aguanta un cambio de semilla. Lo que sí aguanta
es que los árboles no despegan.